# basic

In [4]:
%load_ext autoreload
%autoreload all

In [17]:
import polars as pl
import numpy as np
import pickle
import itertools
import networkx as nx

import src.configs_mapped as configs_mapped
from src.graph_fct import build_relations_graph, get_subgraphs_from_nodes
import src.propagate as propagate

# from src.tokenizer import GraphTokenizer
# from src.iterative_selection import iterative_approach_w_graph_red, IterativeMarginalGainMax

 A concept reaches a flagged candidate not just through IS_A generalization but through any FINDING_SITE/METHOD/COMPONENT/etc. edge in the mix. That's semantically a much looser notion of "covered" than what you actually want (which sounds like: reachable via a specific, meaningful chain — e.g. [some relation] -> IS_A* — not an arbitrary walk through attribute space).

2. No distance decay in propagate() itself. This is the one that explains the "most children = 0.93 at k=1" result more than #1 does. propagate()'s mean-of-out-edges, applied transitively through a topological sort, has no hop penalty at all — a value resolved 5 hops away flows into an intermediate node's average exactly as strongly as one resolved 1 hop away (it only gets diluted if that intermediate node has other competing sources, not because of distance itself). Combined with IS_A's structure — a handful of top concepts sit a small number of hops above enormous descendant subtrees — that's sufficient on its own to let one generic ancestor dominate, even with zero relation-type mixing. Your existing dist_score = 1/(1+distance) in iterative_selection.py:130 already encodes the fix for that candidate-selection objective, but it's not connected to how propagate() computes sem_cov — the two live in separate, inconsistent scoring worlds.

# prepare graph

In [6]:
df_relation = pl.read_parquet(f"{configs_mapped.GraphConfig().relation_path}")
df_concept_path = pl.read_parquet(f"{configs_mapped.GraphConfig().concept_path}")


In [7]:
# 1. get mapped concepts
mapped_ids = df_concept_path.filter(pl.col("is_mapped"))["id"].to_list()
len(mapped_ids)

46150

In [8]:
# 0. build 3 hops subgraph from mapped concepts, regardless type of relations
G = build_relations_graph(df_relation)
subgraphs = get_subgraphs_from_nodes(G, mapped_ids, max_distance=configs_mapped.TokenizerParam().max_dist_candidate)
combined = nx.compose_all([sg for sg in subgraphs.values() if sg is not None])

# 1. build subgraph relation table
df_relations_subgraph = pl.DataFrame(
    [(u, v, d["relation"]) for u, v, d in combined.edges(data=True)],
    schema=["src.id", "dst.id", "relation"],
    orient="row",
)
df_direct = df_relations_subgraph.with_columns(distance = 1).unique()
# df_self = (df_relations_subgraph
#            .select("src.id")
#            .with_columns(distance = 0,
#                            relation = pl.lit("IS_A"))
#              .with_columns(pl.col("src.id").alias("dst.id"))
#              .select(df_direct.columns)
#              .unique())
# mapped_rel_final = pl.concat([df_direct, df_self])

In [9]:
df_direct

src.id,dst.id,relation,distance
str,str,str,i32
"""29689003""","""95880003""","""IS_A""",1
"""117033003:116686009=168139001,…","""410607006""","""COMPONENT""",1
"""737669009""","""11555005""","""IS_A""",1
"""707606004""","""18718003""","""IS_A""",1
"""244436004""","""128319008""","""IS_A""",1
…,…,…,…
"""117033003:116686009=258520000,…","""129265001""","""METHOD""",1
"""117033003:116686009=472903000,…","""129265001""","""METHOD""",1
"""241655007""","""70258002""","""PROCEDURE_SITE_DIRECT""",1


# prepare input for propagation and sem coverage

In [64]:
# 1. find concept propagation depart
cpt_sources = list(set(df_direct["dst.id"].to_list())-set(df_direct["src.id"].to_list()))

# 2. build the edges
edges = []
for row in df_direct.iter_rows(named=True):
    u = row["src.id"]
    v = row["dst.id"]
    edges.append((u,v))

# pick a candidate set and flag
# candidate_test = pl.read_parquet(f"{configs_mapped.IterativeMarginalGain().path}_inv.parquet")["candidate_id"].to_list()
random_k = pl.read_parquet(f"{configs_mapped.Baselines().path}k_random_all_samples.parquet")

import numpy as np
from tqdm  import tqdm

Ks = np.arange(500, 12000, 500)
iters = np.arange(0,50,1)

for k in Ks:
    score_mean = 0
    for iter in tqdm(iters):

        candidate_test = random_k.filter((pl.col("k") == k) & (pl.col("iter") == iter))["candidate_id"]

        flags = dict()
        for cpt_source in cpt_sources:
            flags[cpt_source] = 0 # by default setting becasue they shouls have a value
        for candidate in candidate_test: 
            flags[candidate] = 1 # flag candidates to be 1


        result = propagate.propagate_n_get_value(edges, flags)
        df_result = pl.DataFrame({"concept": list(result.keys()), "sem_cov": [float(v) for v in result.values()]})
        cov = propagate.get_sem_cov_score(result, mapped_ids)
        score_mean += cov['sem_cov'].mean()
    score_mean /= len(iters)
    print(f"Number of selected condidate: {k}, sem_cov score: {score_mean}")
    print(len(cov), "of", len(mapped_ids), "mapped concepts resolved")

100%|██████████| 50/50 [00:49<00:00,  1.02it/s]


Number of selected condidate: 500, sem_cov score: 0.03782664889809152
46150 of 46150 mapped concepts resolved


100%|██████████| 50/50 [00:47<00:00,  1.05it/s]


Number of selected condidate: 1000, sem_cov score: 0.06974702334661437
46150 of 46150 mapped concepts resolved


100%|██████████| 50/50 [00:47<00:00,  1.06it/s]


...

Number of selected condidate: 10500, sem_cov score: 0.6144729395174179
46150 of 46150 mapped concepts resolved


100%|██████████| 50/50 [00:47<00:00,  1.06it/s]


Number of selected condidate: 11000, sem_cov score: 0.6642905535717943
46150 of 46150 mapped concepts resolved


100%|██████████| 50/50 [00:45<00:00,  1.10it/s]

Number of selected condidate: 11500, sem_cov score: 0.7035825667707123
46150 of 46150 mapped concepts resolved


In [66]:
# 1. find concept propagation depart
cpt_sources = list(set(df_direct["dst.id"].to_list())-set(df_direct["src.id"].to_list()))

# 2. build the edges
edges = []
for row in df_direct.iter_rows(named=True):
    u = row["src.id"]
    v = row["dst.id"]
    edges.append((u,v))

# pick a candidate set and flag
candidate_test = pl.read_parquet(f"{configs_mapped.IterativeMarginalGain().path}_inv.parquet")["candidate_id"].to_list()

import numpy as np
from tqdm  import tqdm

Ks = np.arange(1, 1000, 10)

for k in Ks:
    score_mean = 0
    flags = dict()
    for cpt_source in cpt_sources:
        flags[cpt_source] = 0 # by default setting becasue they shouls have a value
    for candidate in candidate_test[:k]: 
        flags[candidate] = 1 # flag candidates to be 1


    result = propagate.propagate_n_get_value(edges, flags)
    df_result = pl.DataFrame({"concept": list(result.keys()), "sem_cov": [float(v) for v in result.values()]})
    cov = propagate.get_sem_cov_score(result, mapped_ids)

    print(f"Number of selected condidate: {k}, sem_cov score: {cov['sem_cov'].mean()}")
    # print(len(cov), "of", len(mapped_ids), "mapped concepts resolved")

Number of selected condidate: 1, sem_cov score: 0.13965419087883457
Number of selected condidate: 11, sem_cov score: 0.9302151046400761
Number of selected condidate: 21, sem_cov score: 0.9336645471704393
Number of selected condidate: 31, sem_cov score: 0.9358153450335096
Number of selected condidate: 41, sem_cov score: 0.9412509105271861
Number of selected condidate: 51, sem_cov score: 0.9412510104523034
Number of selected condidate: 61, sem_cov score: 0.9412510104523035
Number of selected condidate: 71, sem_cov score: 0.9447010406728
Number of selected condidate: 81, sem_cov score: 0.9460558569420978
Number of selected condidate: 91, sem_cov score: 0.9475329636041313
Number of selected condidate: 101, sem_cov score: 0.9493348580315499
Number of selected condidate: 111, sem_cov score: 0.9504776375125867
Number of selected condidate: 121, sem_cov score: 0.9652984563398053
Number of selected condidate: 131, sem_cov score: 0.9663898942207639
Number of selected condidate: 141, sem_cov scor

In [67]:
# 1. find concept propagation depart
cpt_sources = list(set(df_direct["dst.id"].to_list())-set(df_direct["src.id"].to_list()))

# 2. build the edges
edges = []
for row in df_direct.iter_rows(named=True):
    u = row["src.id"]
    v = row["dst.id"]
    edges.append((u,v))

# pick a candidate set and flag
candidate_test = pl.read_parquet(f"{configs_mapped.Baselines().path}highest_degree_dist_1.parquet")["dst.id"].to_list()

import numpy as np
from tqdm  import tqdm

Ks = np.arange(1, 1000, 10)

for k in Ks:
    score_mean = 0
    flags = dict()
    for cpt_source in cpt_sources:
        flags[cpt_source] = 0 # by default setting becasue they shouls have a value
    for candidate in candidate_test[:k]: 
        flags[candidate] = 1 # flag candidates to be 1


    result = propagate.propagate_n_get_value(edges, flags)
    df_result = pl.DataFrame({"concept": list(result.keys()), "sem_cov": [float(v) for v in result.values()]})
    cov = propagate.get_sem_cov_score(result, mapped_ids)

    print(f"Number of selected condidate: {k}, sem_cov score: {cov['sem_cov'].mean()}")
    # print(len(cov), "of", len(mapped_ids), "mapped concepts resolved")

Number of selected condidate: 1, sem_cov score: 0.13965419087883457
Number of selected condidate: 11, sem_cov score: 0.2587037833115191
Number of selected condidate: 21, sem_cov score: 0.2979020386364036
Number of selected condidate: 31, sem_cov score: 0.3201515387551066
Number of selected condidate: 41, sem_cov score: 0.3460972195816468
Number of selected condidate: 51, sem_cov score: 0.3676244142898715
Number of selected condidate: 61, sem_cov score: 0.3810596047252316
Number of selected condidate: 71, sem_cov score: 0.4927117409117751
Number of selected condidate: 81, sem_cov score: 0.5069407332727441
Number of selected condidate: 91, sem_cov score: 0.6282109809792753
Number of selected condidate: 101, sem_cov score: 0.6333565185889879
Number of selected condidate: 111, sem_cov score: 0.638817247646561
Number of selected condidate: 121, sem_cov score: 0.6676591113920071
Number of selected condidate: 131, sem_cov score: 0.6954715717236019
Number of selected condidate: 141, sem_cov sc

In [68]:

# 1. find concept propagation depart
cpt_sources = list(set(df_direct["dst.id"].to_list())-set(df_direct["src.id"].to_list()))

# 2. build the edges
edges = []
for row in df_direct.iter_rows(named=True):
    u = row["src.id"]
    v = row["dst.id"]
    edges.append((u,v))

# pick a candidate set and flag
candidate_test = pl.read_parquet(f"{configs_mapped.Baselines().path}most_children.parquet")["candidate"].to_list()

import numpy as np
from tqdm  import tqdm

Ks = np.arange(1, 1000, 10)

for k in Ks:
    score_mean = 0
    flags = dict()
    for cpt_source in cpt_sources:
        flags[cpt_source] = 0 # by default setting becasue they shouls have a value
    for candidate in candidate_test[:k]: 
        flags[candidate] = 1 # flag candidates to be 1


    result = propagate.propagate_n_get_value(edges, flags)
    df_result = pl.DataFrame({"concept": list(result.keys()), "sem_cov": [float(v) for v in result.values()]})
    cov = propagate.get_sem_cov_score(result, mapped_ids)

    print(f"Number of selected condidate: {k}, sem_cov score: {cov['sem_cov'].mean()}")
    # print(len(cov), "of", len(mapped_ids), "mapped concepts resolved")

Number of selected condidate: 1, sem_cov score: 0.9302151046400761
Number of selected condidate: 11, sem_cov score: 0.9302151046400761
Number of selected condidate: 21, sem_cov score: 0.9302151046400761
Number of selected condidate: 31, sem_cov score: 0.9302151046400761
Number of selected condidate: 41, sem_cov score: 0.9306595661699376
Number of selected condidate: 51, sem_cov score: 0.9309377248844644
Number of selected condidate: 61, sem_cov score: 0.9309377248844642
Number of selected condidate: 71, sem_cov score: 0.9312847818976496
Number of selected condidate: 81, sem_cov score: 0.9434902869788093
Number of selected condidate: 91, sem_cov score: 0.9434902869788093
Number of selected condidate: 101, sem_cov score: 0.9434902869788095
Number of selected condidate: 111, sem_cov score: 0.9434902869788095
Number of selected condidate: 121, sem_cov score: 0.9434902869788093
Number of selected condidate: 131, sem_cov score: 0.9434902869788093
Number of selected condidate: 141, sem_cov sc